In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import plotly.graph_objects as go
from torch.utils.data import TensorDataset, DataLoader
from typing import Callable
from tqdm.notebook import tqdm

In [ ]:
torch.manual_seed(42)
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device:", device)

To generate my data (points on a circle) I sample the angle $\theta$ from a uniform in $[0,2\pi]$ and use polar coordinates from a radius $r$:
- $x = r\cdot cos(\theta)$
- $y = r\cdot sin(\theta)$

In [ ]:
# initial dataset and data loader
def generate_circle_data(num_samples=5000, radius=2.0):
    theta = torch.rand(num_samples) * 2 * math.pi
    x = radius * torch.cos(theta)
    y = radius * torch.sin(theta)
    return torch.stack((x, y), dim=-1)

x_clean_train = generate_circle_data(num_samples=10000)
train_ds = TensorDataset(x_clean_train)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True) 

### ARCHITECTURE
Input: a point $(x,y)$

Output: the predicted noise $(\epsilon_x, \epsilon_y)$

From the predicted noise we're defining a direction because when we substract it we move closer to the clean data.

In [ ]:

class ScoreMLP2D(nn.Module):
    def __init__(self, 
                 num_layers: int, 
                 hidden_dim: int,
                 activation: Callable[[torch.Tensor], torch.Tensor]) -> None:
        super().__init__()
        self.first_layer = nn.Linear(in_features=2, out_features=hidden_dim)
        self.layers = nn.ModuleList() 
        for i in range(num_layers):
            self.layers.append(
                nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
            )
        self.activation = activation
        self.last_layer = nn.Linear(in_features=hidden_dim, out_features=2)

    def forward(self, meshgrid: torch.Tensor) -> torch.Tensor:
        out = meshgrid
        out = self.first_layer(out)
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
        out = self.last_layer(out)
        return out



In [ ]:
#initial state and training parameters
model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
sigma = 0.3 
epochs = 6#50

In [ ]:
for epoch in tqdm(range(epochs), desc="epoch"):
    model.train()
    epoch_loss = 0.0  
    num_batches = 0   
    for (xb,) in train_dl:
        x_clean = xb.to(device)
        noise = torch.randn_like(x_clean)
        # here I dirty the data with noise
        x_noisy = x_clean + sigma * noise
        pred_noise = model(x_noisy)
        #I compute the mse loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item()
        num_batches += 1
        
    # mean loss in epoch
    print(epoch, epoch_loss / num_batches)
    #if (epoch + 1) % 10 == 0 or epoch == 0:
    #    print(epoch, epoch_loss / num_batches)

Given the equations in the pdf I can get the score as a function of the noise.

- $\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{1}{\sigma^2}(\mathbf{x} - \tilde{\mathbf{x}})$

- $\tilde{\mathbf{x}} = \mathbf{x} + \sigma \epsilon \implies \mathbf{x} - \tilde{\mathbf{x}} = -\sigma \epsilon$

Here:
* **$\mathbf{x}$** is the original data
* **$\tilde{\mathbf{x}}$** represents noisy data
* **$\sigma$** is the SD of the Gaussian noise
* **$\epsilon$** is the standard Gaussian noise vector that will dirty my data
* **$q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x})$** is the conditional probability distribution of the noisy data given the clean data and will model the noise corruption process.
* **$\nabla_{\tilde{\mathbf{x}}}$**: the gradient is computed wrt the noisy data and provides the direction of change

In the end I get
$$\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{-\sigma \epsilon}{\sigma^2} = -\frac{\epsilon}{\sigma}$$

Then I implement Langevin dynamics `x_langevin` from the following equation
$$\tilde{\mathbf{x}}^k = \tilde{\mathbf{x}}^{k-1} + \frac{\lambda_i}{2} \mathbf{s}_\theta(\tilde{\mathbf{x}}^{k-1}, \sigma_i) + \sqrt{\lambda_i}\mathbf{z}^k$$

where at each step $k$ I update the previous position $\tilde{\mathbf{x}}^{k-1}$ moving along the direction of the score with a step length scaled by $\lambda_i$. A standard Gaussian vector $\mathbf{z}^k$ adds randomness to explore the space.

In [87]:
# langevin sampling with snapshots
model.eval()
num_steps = 300
step_size = 0.01  

x_langevin = torch.randn(1000, 2, device=device) * 4.0 
initial_noise_np = x_langevin.cpu().clone().numpy()

snapshots = {}
monitored_steps = [1, 10, 20, 50, 100, 300]

with torch.no_grad():
    for k in range(num_steps):
        pred_noise = model(x_langevin)
        score = - pred_noise / sigma
        
        # langevin dynamics step
        z = torch.randn_like(x_langevin)
        x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z
        
        current_step = k + 1
        if current_step in monitored_steps:
            snapshots[current_step] = x_langevin.cpu().numpy()

final_generated_np = x_langevin.cpu().numpy()

In [88]:
#visualization
x_clean_np = x_clean_train.cpu().numpy()
def plot_2d_points(points, title, color):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=4, color=color, opacity=0.6),
            name=title
        )
    )
    fig.update_layout(
        title=title, width=600, height=600,
        xaxis=dict(range=[-6, 6]), yaxis=dict(range=[-6, 6])
    )
    fig.show()

plot_2d_points(x_clean_np, "Original data", "lightgreen")
plot_2d_points(initial_noise_np, "Noise", "red")
plot_2d_points(final_generated_np, "Data generated", "orange")

In [89]:
#data generation process
plot_2d_points(initial_noise_np, "Step 0: Pure Noise", "red")
for step in monitored_steps:
    plot_2d_points(
        snapshots[step], 
        title=f"Step {step}: Inversion Process", 
        color="orange"
    )

In [90]:
#better representation in a grid
from plotly.subplots import make_subplots
import plotly.graph_objects as go

all_steps = monitored_steps
titles = [f"Step {step}" for step in all_steps]

fig = make_subplots(rows=2, cols=3, subplot_titles=titles)

for idx, step in enumerate(all_steps):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    points = initial_noise_np if step == 0 else snapshots[step]
    
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=3, color="orange", opacity=0.6),
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_layout(width=1050, height=700, template="plotly_white")
fig.update_xaxes(range=[-6, 6])
fig.update_yaxes(range=[-6, 6])

fig.show()

Let's see if it generalizes: let's generate sinusoids

In [91]:
def generate_sine_data(num_samples=5000, amp=1.0, freq=1.0):
    theta = torch.rand(num_samples) * 2 * math.pi
    y = amp * torch.sin(freq * theta)
    
    return torch.stack((theta, y), dim=-1)

x_clean_sine = generate_sine_data(num_samples=10000)
print(x_clean_sine.shape)  
train_ds_sine = TensorDataset(x_clean_sine)
train_dl_sine = DataLoader(train_ds_sine, batch_size=64, shuffle=True) 

torch.Size([10000, 2])


In [ ]:
#better representation in a grid
from plotly.subplots import make_subplots
import plotly.graph_objects as go
#initial state and training parameters
model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
sigma = 0.3 
epochs = 21
losses_h = []
for epoch in tqdm(range(epochs), desc="epoch"):
    model.train()
    epoch_loss = 0.0  
    num_batches = 0   
    for (xb,) in train_dl_sine:
        x_clean = xb.to(device)
        noise = torch.randn_like(x_clean)
        # here I dirty the data with noise
        x_noisy = x_clean + sigma * noise
        pred_noise = model(x_noisy)
        #I compute the mse loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item()
        num_batches += 1
        
    # mean loss in epoch
    mean_epoch_loss = epoch_loss / num_batches
    losses_h.append(mean_epoch_loss)
    print(epoch, mean_epoch_loss)
    #if (epoch + 1) % 10 == 0 or epoch == 0:
    #    print(epoch, epoch_loss / num_batches)
    
# just some monitoring
min_loss_index = losses_h.index(min(losses_h))
print(f"Minimum loss of {losses_h[min_loss_index]:.6f} at epoch {min_loss_index}")
#loss plot
loss_fig = go.Figure()
loss_fig.add_trace(
    go.Scatter(
        x=list(range(epochs)), 
        y=losses_h, 
        mode='lines+markers', 
        name='MSE Loss',
        line=dict(color='deepskyblue', width=2)
    )
)
loss_fig.update_layout(
    title="Training Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss",
    width=700, height=400,
    template="plotly_white"
)
loss_fig.show()

# langevin sampling with snapshots
model.eval()
num_steps = 1000
step_size = 0.01  

x_langevin = torch.randn(1000, 2, device=device) * 4.0 
initial_noise_np = x_langevin.cpu().clone().numpy()

snapshots = {}
monitored_steps = [1, 10, 20, 50, 100, 300, 500, 1000]

with torch.no_grad():
    for k in range(num_steps):
        pred_noise = model(x_langevin)
        score = - pred_noise / sigma
        
        # langevin dynamics step
        z = torch.randn_like(x_langevin)
        x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z
        
        current_step = k + 1
        if current_step in monitored_steps:
            snapshots[current_step] = x_langevin.cpu().numpy()

final_generated_np = x_langevin.cpu().numpy()
all_steps =  monitored_steps
titles = [f"Step {step}" for step in all_steps]

fig = make_subplots(rows=2, cols=4, subplot_titles=titles)

for idx, step in enumerate(all_steps):
    row = (idx // 4) + 1
    col = (idx % 4) + 1
    points = initial_noise_np if step == 0 else snapshots[step]
    
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=3, color="orange", opacity=0.6),
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_layout(width=1050, height=700, template="plotly_white")
fig.update_xaxes(range=[-6, 6])
fig.update_yaxes(range=[-6, 6])

fig.show()

epoch:   0%|          | 0/21 [00:00<?, ?it/s]

0 0.6383942238464477
1 0.5372262228826049
2 0.5210222060892992
3 0.5035342276096344
4 0.4907106138338709
5 0.48712278247638874
6 0.48231865835797255
7 0.46971647090213314
8 0.4821758348091393
9 0.4844836102929085
10 0.47365540987367083
11 0.4783300926351243
12 0.47929721634099437
13 0.4656571087184226
14 0.47147470750626486
15 0.47289693260648447
16 0.46386582437594226
17 0.46966662490443817
18 0.47799588787327907
19 0.46802489469005804
20 0.474732899741762
Minimum loss of 0.463866 at epoch 16


## With different values of $\sigma$

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from tqdm.notebook import tqdm
import torch.nn.functional as F
import torch.optim as optim
import torch
import math

sigmas = [0.1, 0.3, 0.8]
epochs = 21

for sigma in sigmas:
    print("\n" + "="*50)
    print(f" RUNNING EXPERIMENT WITH SIGMA = {sigma} ")
    print("="*50)
    
    # init
    model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=0.001)
    
    losses_h = []
    
    # training
    for epoch in tqdm(range(epochs), desc=f"Training (sigma={sigma})"):
        model.train()
        epoch_loss = 0.0  
        num_batches = 0   
        for (xb,) in train_dl_sine:
            x_clean = xb.to(device)
            noise = torch.randn_like(x_clean)
            x_noisy = x_clean + sigma * noise
            pred_noise = model(x_noisy)
            
            loss = F.mse_loss(pred_noise, noise)
            loss.backward()
            opt.step()
            opt.zero_grad()
            
            epoch_loss += loss.item()
            num_batches += 1
            
        mean_epoch_loss = epoch_loss / num_batches
        losses_h.append(mean_epoch_loss)
        
    min_loss_index = losses_h.index(min(losses_h))
    print(f"[Sigma {sigma}] min loss {losses_h[min_loss_index]:.6f} at epoch {min_loss_index}")
    
    # training loss plot
    loss_fig = go.Figure()
    loss_fig.add_trace(
        go.Scatter(
            x=list(range(epochs)), 
            y=losses_h, 
            mode='lines+markers', 
            name=f'MSE Loss (sigma={sigma})',
            line=dict(color='deepskyblue', width=2)
        )
    )
    loss_fig.update_layout(
        title=f"Training Loss Curve (Sigma = {sigma})",
        xaxis_title="Epoch", yaxis_title="MSE Loss",
        width=700, height=350, template="plotly_white"
    )
    loss_fig.show()
    
    # langevin sampling
    model.eval()
    num_steps = 1000
    step_size = 0.01  
    
    x_langevin = torch.randn(1000, 2, device=device) * 4.0 
    initial_noise_np = x_langevin.cpu().clone().numpy()
    
    snapshots = {}
    monitored_steps = [1, 10, 20, 50, 100, 300, 500, 1000]
    
    with torch.no_grad():
        for k in range(num_steps):
            pred_noise = model(x_langevin)
            score = - pred_noise / sigma
            
            z = torch.randn_like(x_langevin)
            x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z
            
            current_step = k + 1
            if current_step in monitored_steps:
                snapshots[current_step] = x_langevin.cpu().numpy()
    
    all_steps = monitored_steps
    titles = [f"Step {step}" for step in all_steps]
    grid_fig = make_subplots(rows=2, cols=4, subplot_titles=titles)
    
    for idx, step in enumerate(all_steps):
        row = (idx // 4) + 1
        col = (idx % 4) + 1
        points = initial_noise_np if step == 0 else snapshots[step]
        
        grid_fig.add_trace(
            go.Scatter(
                x=points[:, 0], y=points[:, 1],
                mode='markers',
                marker=dict(size=3, color="orange", opacity=0.6),
                showlegend=False
            ),
            row=row, col=col
        )
        
    grid_fig.update_layout(
        title_text=f"Langevin Inversion Grid (Sigma = {sigma})",
        width=1050, height=600, template="plotly_white"
    )
    grid_fig.update_xaxes(range=[-6, 6])
    grid_fig.update_yaxes(range=[-6, 6])
    grid_fig.show()


 RUNNING EXPERIMENT WITH SIGMA = 0.1 


Training (sigma=0.1):   0%|          | 0/21 [00:00<?, ?it/s]